In [1]:
import pandas as pd
import numpy as np

def data_harmonization(dc):
    
    aa_dict = {'A': 'Ala','R': 'Arg','N': 'Asn','D': 'Asp','C': 'Cys','E': 'Glu','Q': 'Gln','G': 'Gly','H': 'His',
           'I': 'Ile','L': 'Leu','K': 'Lys','M': 'Met','F': 'Phe','P': 'Pro','S': 'Ser','T': 'Thr','W': 'Trp',
           'Y': 'Tyr','V': 'Val', '=': '=','*': 'Ter', "~": 'Del', '-':'Del'}

    reversed_aa_dict = {v: k for k, v in aa_dict.items()}
    
    x = dc['x']
    
    y = dc['y']
    
    df1 = pd.read_excel(x, header = y)
    
    #create an empty dataframe with column names as follows

    pillar_data = pd.DataFrame(columns=["Dataset", "Gene", "HGNC_id","Chrom","hg19_pos","hg38_pos","ref_allele",
                                    "alt_allele","auth_transcript_id","transcript_pos","transcript_ref","transcript_alt",
                                   "aa_pos","aa_ref","aa_alt","hgvs_c","hgvs_p","consequence","auth_reported_score",
                                   "auth_reported_rep_score","auth_reported_func_class","auth_reported_normal_min",
                                   "auth_reported_normal_max","auth_reported_abnormal_min","auth_reported_abnormal_max",
                                   "splice_measure","gnomad_MAF","clinvar_sig","clinvar_star",
                                    "clinvar_date_last_reviewed","nucleotide_or_aa"])
    
    num_rows = len(df1)
    pillar_data = pd.DataFrame(index=range(num_rows), columns=pillar_data.columns)

    for key, value in dc.items():
        if key in pillar_data.columns:
            if pd.api.types.is_scalar(value):
                pillar_data[key] = value
            else:
                pillar_data[key] = value.values
                    
                    
    #converting syntax such as p.M1L or M1L to p.Met1Leu
    if dc['hgvs_p_conversion'] == 'Yes':
        hgvs_pro_list = list()
        for index, row in pillar_data.iterrows():
            try:
                if pd.isna(row['hgvs_p']):
                    hgvs_pro = np.nan
                    hgvs_pro_list.append(hgvs_pro)
                elif row['hgvs_p'].startswith('p.'):
                    if row['hgvs_p'].endswith("="):
                        alt_a = row["hgvs_p"][2]
                        ref_a = row['hgvs_p'][2]
                        position = row['hgvs_p'][3:-1]
                    else:
                        ref_a = row['hgvs_p'][2]
                        alt_a = row['hgvs_p'][-1]
                        position = row['hgvs_p'][3:-1]
                    hgvs_pro = f"p.{aa_dict[ref_a]}{position}{aa_dict[alt_a]}"
                    hgvs_pro_list.append(hgvs_pro)
                else:
                    if row['hgvs_p'].endswith("="):
                        alt_a = row["hgvs_p"][2]
                        ref_a = row['hgvs_p'][0]
                        position = row['hgvs_p'][1:-1]
                    else:
                        ref_a = row['hgvs_p'][0]
                        alt_a = row['hgvs_p'][-1]
                        position = row['hgvs_p'][1:-1]
                    hgvs_pro = f"p.{aa_dict[ref_a]}{position}{aa_dict[alt_a]}"
                    hgvs_pro_list.append(hgvs_pro)
                    
            except KeyError as e:
                print(f"KeyError for row {index} with value {row['hgvs_p']}: {e}, {x}")
                hgvs_pro_list.append(row['hgvs_p'])
                
                continue

        pillar_data['hgvs_p'] = hgvs_pro_list
        
        

    #uses amino acid pos, ref, and alt to construct an hgvs p. format. 
    if dc['hgvs_from_aa'] == 'Yes':
        hgvs_pro_list = list()
        for index, row in pillar_data.iterrows():
            if pd.isna(row['aa_ref']) & pd.isna(row['aa_alt']):
                hgvs_pro = np.nan
                hgvs_pro_list.append(hgvs_pro)
            else:
                hgvs_pro_list.append(f"p.{aa_dict[row['aa_ref']]}{row['aa_pos']}{aa_dict[row['aa_alt']]}")
                
        pillar_data['hgvs_p'] = hgvs_pro_list

    #uses hgvs p. format to fill in amino acid pos, alt and ref columns
    if dc['aa_from_hgvs'] == 'Yes':
        aa_ref = list()
        aa_alt = list()
        aa_pos = list()
        for index, row in pillar_data.iterrows():
            try:
                if pd.isna(row['hgvs_p']):
                    aa_ref.append(np.nan) 
                    aa_alt.append(np.nan)
                    aa_pos.append(np.nan)
                else:
                    aa_ref.append(reversed_aa_dict[row['hgvs_p'][2:5]])
                    if row['hgvs_p'].endswith("="):
                        aa_alt.append('=')
                        aa_pos.append(row['hgvs_p'][5:-1])
                    elif row['hgvs_p'].endswith("*"):
                        aa_alt.append('*')
                        aa_pos.append(row['hgvs_p'][5:-1])
                    elif row['hgvs_p'].endswith("Ter"):
                        aa_alt.append('*')
                        aa_pos.append(row['hgvs_p'][5:-3])
                    else:
                        aa_alt.append(reversed_aa_dict[row['hgvs_p'][-3:]])
                        aa_pos.append(row['hgvs_p'][5:-3])
                        
            except KeyError as e:
                print(f"KeyError for row {index} with value {row['hgvs_p']}: {e}, {x}")
                aa_ref.append(row['hgvs_p'])
                aa_alt.append(row['hgvs_p'])
                aa_pos.append(np.nan)
                
                continue
                
        pillar_data['aa_ref'] = aa_ref
        pillar_data['aa_alt'] = aa_alt
        pillar_data['aa_pos'] = aa_pos
    
    #if hgvs_c is provided, it uses that information to populate transcript information
    if dc['transcript_from_hgvs_c'] == 'Yes':
        transcript_pos = list()
        transcript_ref = list()
        transcript_alt = list()
        for index, row in pillar_data.iterrows():
            if pd.isna(row['hgvs_c']):
                transcript_ref.append(np.nan) 
                transcript_alt.append(np.nan)
                transcript_pos.append(np.nan)
            else:
                transcript_ref.append(row['hgvs_c'][-3])
                transcript_alt.append(row['hgvs_c'][-1])
                transcript_pos.append(row['hgvs_c'][2:-3])
        pillar_data['transcript_pos'] = transcript_pos
        pillar_data['transcript_ref'] = transcript_ref
        pillar_data['transcript_alt'] = transcript_alt
                               
                
    return(pillar_data)

In [2]:
def data_harmonization_loop(dc_list):
    
    combined_dataframes = []
    
    
    for dc in dc_list:
            harmonized_df = data_harmonization(dc)
        
        
            if harmonized_df is not None and not harmonized_df.empty:
                combined_dataframes.append(harmonized_df) 
    
    
    combined_df = pd.concat(combined_dataframes, ignore_index=True) if combined_dataframes else pd.DataFrame()
    
    return combined_df

In [3]:
df1 = pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA1_Findlay_2018.xlsx", header = 2)
dc_1 = {"x": "~/Downloads/Pillar_project_data_files/BRCA1_Findlay_2018.xlsx", 
               "y": 2,
               "Dataset" :'BRCA1_Findlay_2018', 
               "Gene" : df1['gene'], 
               "HGNC_id" : 1100, 
               "Chrom" : df1['chromosome'], 
               "hg19_pos" : df1['position (hg19)'], 
               "hg38_pos" : np.nan, 
               "ref_allele": df1['reference'], 
               "alt_allele": df1['alt'], 
               "auth_transcript_id" : df1['transcript_ID'], 
               "transcript_pos" : df1['transcript_position'],
               "transcript_ref" : df1['transcript_ref'], 
               "transcript_alt" : df1['transcript_alt'],
               "aa_pos": df1['aa_pos'],
               "aa_ref": df1['aa_ref'],
               "aa_alt": df1['aa_alt'],
               "hgvs_c": df1['transcript_variant'],
               "hgvs_p" : df1['protein_variant'],
               "consequence": df1['consequence'],
               "auth_reported_score": df1['function.score.mean'],
               "auth_reported_rep_score": df1[['function.score.r1', 'function.score.r2']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": df1['func.class'],
               "auth_reported_normal_min":-0.748,
               "auth_reported_normal_max":1.307,
               "auth_reported_abnormal_min":-5.651,
               "auth_reported_abnormal_max":-1.328,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#one amino acid change from reference sequence; amino acid 1613 in reference changed from S-->G in experiment
df2 = pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA1_Adamovich_2022_HDR.xlsx", header = 0)
dc_2 = {"x": "~/Downloads/Pillar_project_data_files/BRCA1_Adamovich_2022_HDR.xlsx", 
               "y": 0,
               "Dataset" :'BRCA1_Adamovich_2022_HDR', 
               "Gene" : "BRCA1", 
               "HGNC_id" : 1100, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df2['variantID'],
               "consequence": np.nan,
               "auth_reported_score": df2['FS_average_BRCA1_siRNA'],
               "auth_reported_rep_score": df2[['FS1_BRCA1_siRNA', 'FS2_BRCA1_siRNA','FS3_BRCA1_siRNA','FS4_BRCA1_siRNA']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min":-0.233,
               "auth_reported_normal_max":1.357,
               "auth_reported_abnormal_min":-3.198,
               "auth_reported_abnormal_max":-0.562,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#one amino acid change from reference sequence; amino acid 1613 in reference changed from S-->G in experiment
df3 = pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA1_Adamovich_2022_Cisplatin.xlsx", header = 0)
dc_3 = {"x": "~/Downloads/Pillar_project_data_files/BRCA1_Adamovich_2022_Cisplatin.xlsx", 
               "y": 0,
               "Dataset" :'BRCA1_Adamovich_2022_Cisplatin', 
               "Gene" : "BRCA1", 
               "HGNC_id" : 1100, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df3['variantID'],
               "consequence": np.nan,
               "auth_reported_score": df3['FS_average_BRCA1_siRNA'],
               "auth_reported_rep_score": df3[['FS1_BRCA1_siRNA', 'FS2_BRCA1_siRNA','FS3_BRCA1_siRNA','FS4_BRCA1_siRNA']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min":-0.503,
               "auth_reported_normal_max":0.942,
               "auth_reported_abnormal_min":-3.198,
               "auth_reported_abnormal_max":-0.729,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#p.Val2687A changed to p.Val2687Ala. Mistake in file from authors
df4 = pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA2_Hu_2024.xlsx", header = 2)
dc_4 = {"x": "~/Downloads/Pillar_project_data_files/BRCA2_Hu_2024.xlsx", 
               "y": 2,
               "Dataset" :'BRCA2_Hu_2024', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.3", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df4['coding nucleotide change'],
               "hgvs_p" : df4['Protein change'],
               "consequence": np.nan,
               "auth_reported_score": df4['HDR score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df4['HDR function'],
               "auth_reported_normal_min":2.5,
               "auth_reported_normal_max":np.nan,
               "auth_reported_abnormal_min":np.nan,
               "auth_reported_abnormal_max":1.49,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

df5 = pd.read_excel("~/Downloads/Pillar_project_data_files/MSH2_Jia_2021.xlsx", header = 0)
dc_5 = {"x": "~/Downloads/Pillar_project_data_files/MSH2_Jia_2021.xlsx", 
               "y": 0,
               "Dataset" :'MSH2_Jia_2021', 
               "Gene" : "MSH2", 
               "HGNC_id" : 7325, 
               "Chrom" : 2, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000251.2", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df5["Position"],
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df5['Variant'],
               "consequence": np.nan,
               "auth_reported_score": df5['LOF score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": 0,
               "auth_reported_abnormal_min": 0,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df6 = pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_6 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_WAF1nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df6['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df6['WAF1nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df7 = pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_7 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_MDM2nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df7['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df7['MDM2nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}
df8 = pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_8 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_BAXnWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df8['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df8['BAXnWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}
df9= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_9 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_h1433snWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df9['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df9['h1433snWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df10= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_10 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_AIP1nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df10['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df10['AIP1nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df11= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_11 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_GADD45nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df11['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df11['GADD45nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df12= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_12 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_NOXAnWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df12['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df12['NOXAnWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df13= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", header = 0)
dc_13 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Kato_2003.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Kato_2003_P53R2nWT', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df13['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df13['P53R2nWT'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df14= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Fortuno_2021_Kato_meta.xlsx", header = 0)
dc_14 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Fortuno_2021_Kato_meta.xlsx", 
               "y": 0,
               "Dataset" :'TP53_Fortuno_2021_Kato_meta', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : 'NM_000546.5', 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df14['ProtDescription'],
               "consequence": np.nan,
               "auth_reported_score": df14['Fortuno_median'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#changed 'B' amino acids to '=' and 'Z' amino acids to '*'; B is synonymous change and Z is nonsense change
df15= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", header = 1)
dc_15 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Giacomelli_2018_p53WT_Nutlin3', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df15["Position"],
               "aa_ref": df15["AA_wt"],
               "aa_alt": df15["AA_variant"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df15['A549_p53WT_Nutlin-3_Z-score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#changed 'B' amino acids to '=' and 'Z' amino acids to '*'; B is synonymous change and Z is nonsense change
df16= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", header = 1)
dc_16 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Giacomelli_2018_p53null_Nutlin3', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df16["Position"],
               "aa_ref": df16["AA_wt"],
               "aa_alt": df16["AA_variant"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df16['A549_p53NULL_Nutlin-3_Z-score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#changed 'B' amino acids to '=' and 'Z' amino acids to '*'; B is synonymous change and Z is nonsense change
df17= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", header = 1)
dc_17 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Giacomelli_2018_p53null_etoposide', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df17["Position"],
               "aa_ref": df17["AA_wt"],
               "aa_alt": df17["AA_variant"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df17['A549_p53NULL_Etoposide_Z-score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#changed 'B' amino acids to '=' and 'Z' amino acids to '*'; B is synonymous change and Z is nonsense change
df18= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", header = 1)
dc_18 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Giacomelli_2018.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Giacomelli_2018_combined_score', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df18["Position"],
               "aa_ref": df18["AA_wt"],
               "aa_alt": df18["AA_variant"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df18['Combined_score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#"Z" amino acids changed to "*", in Variant column where terminating amino acids are changed, Z is changed to "Ter", "B" is changed to "="
df19= pd.read_excel("~/Downloads/Pillar_project_data_files/TP53_Fayer_2021.xlsx", header = 1)
dc_19 = {"x": "~/Downloads/Pillar_project_data_files/TP53_Fayer_2021.xlsx", 
               "y": 1,
               "Dataset" :'TP53_Fayer_2021_meta', 
               "Gene" : "TP53", 
               "HGNC_id" : 11998, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000546.5", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df19["Variant"],
               "consequence": np.nan,
               "auth_reported_score": df19['Classifier_prob_func_abnormal'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df19['Classifier_prediction'],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df20= pd.read_excel("~/Downloads/Pillar_project_data_files/PTEN_Mighell_2018.xlsx", header = 1)
dc_20 = {"x": "~/Downloads/Pillar_project_data_files/PTEN_Mighell_2018.xlsx", 
               "y": 1,
               "Dataset" :'PTEN_Mighell_2018', 
               "Gene" : "PTEN", 
               "HGNC_id" : 9588, 
               "Chrom" : 10, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000314.6", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df20["Variant (one letter)"],
               "consequence": np.nan,
               "auth_reported_score": df20['Cum_score'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df20['High_conf'],
               "auth_reported_normal_min": -1.11,
               "auth_reported_normal_max": 0.89,
               "auth_reported_abnormal_min": -5.76,
               "auth_reported_abnormal_max":-2.13,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

df21= pd.read_excel("~/Downloads/Pillar_project_data_files/PTEN_Matreyek_2018.xlsx", header = 0)
dc_21 = {"x": "~/Downloads/Pillar_project_data_files/PTEN_Matreyek_2018.xlsx", 
               "y": 0,
               "Dataset" :'PTEN_Matreyek_2018', 
               "Gene" : "PTEN", 
               "HGNC_id" : 9588, 
               "Chrom" : 10, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df21["hgvs_pro"],
               "consequence": np.nan,
               "auth_reported_score": df21['score'],
               "auth_reported_rep_score": df21[['score1','score2','score3','score4','score5',
                                                'score6','score7','score8']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": 0.71,
               "auth_reported_normal_max": 1.462,
               "auth_reported_abnormal_min": -0.223,
               "auth_reported_abnormal_max":0.4,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#changed "X" to "*", nonsense
df22= pd.read_excel("~/Downloads/Pillar_project_data_files/SCN5A_Glazer_2020.xlsx", header = 0)
dc_22 = {"x": "~/Downloads/Pillar_project_data_files/SCN5A_Glazer_2020.xlsx", 
               "y": 0,
               "Dataset" :'SCN5A_Glazer_2020', 
               "Gene" : "SCN5A", 
               "HGNC_id" : 10593, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "ENST00000333535", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df22["aa_num"],
               "aa_ref": df22["wt_allele"],
               "aa_alt": df22["mut_allele"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df22['dms'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df22["class"],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

df23= pd.read_excel("~/Downloads/Pillar_project_data_files/VHL_Buckley_2024.xlsx", header = 2)
dc_23 = {"x": "~/Downloads/Pillar_project_data_files/VHL_Buckley_2024.xlsx", 
               "y": 2,
               "Dataset" :'VHL_Buckley_2024', 
               "Gene" : "VHL", 
               "HGNC_id" : 12687, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : df23["hg38_pos"], 
               "ref_allele": df23["ref"], 
               "alt_allele": df23["alt"], 
               "auth_transcript_id" : "ENST00000256474.3", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df23["protPos"],
               "aa_ref": df23["oAA"],
               "aa_alt": df23["nAA"],
               "hgvs_c": df23["cHGVS"],
               "hgvs_p" : df23["pHGVS"],
               "consequence": np.nan,
               "auth_reported_score": df23['function_score_final'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df23["function_class"],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'Yes'}

#mutAA column "X" replaced with "*"
df24= pd.read_excel("~/Downloads/Pillar_project_data_files/KCNH2_Kozek_Glazer_2020.xlsx", header = 1)
dc_24 = {"x": "~/Downloads/Pillar_project_data_files/KCNH2_Kozek_Glazer_2020.xlsx", 
               "y": 1,
               "Dataset" :'KCNH2_Kozek_Glazer_2020', 
               "Gene" : "KCNH2", 
               "HGNC_id" : 6251, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "ENST00000262186", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df24["resnum"],
               "aa_ref": df24["nativeAA"],
               "aa_alt": df24["mutAA"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df24['score.ave'],
               "auth_reported_rep_score": df24[['score.1', 'score.1']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": 75,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":75,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#had to reformat excel file to run through script; reformatting done in excel
df25= pd.read_excel("~/Downloads/Pillar_project_data_files/KCNH2_Jiang_2022.xlsx", header = 0)
dc_25 = {"x": "~/Downloads/Pillar_project_data_files/KCNH2_Jiang_2022.xlsx", 
               "y": 0,
               "Dataset" :'KCNH2_Jiang_2022', 
               "Gene" : "KCNH2", 
               "HGNC_id" : 6251, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000238.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df25["Protein Change"],
               "consequence": np.nan,
               "auth_reported_score": df25[' Normalised Current '],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": 0.78,
               "auth_reported_normal_max": 1.22,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":0.78,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#added 'p.' in the Human variant column
df26= pd.read_excel("~/Downloads/Pillar_project_data_files/OTC_Lo_2023.xlsx", header = 1)
dc_26 = {"x": "~/Downloads/Pillar_project_data_files/OTC_Lo_2023.xlsx", 
               "y": 1,
               "Dataset" :'OTC_Lo_2023', 
               "Gene" : "OTC", 
               "HGNC_id" : 8512, 
               "Chrom" : "X", 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000531.6", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df26["Human_Variant"],
               "consequence": np.nan,
               "auth_reported_score": df26['Growth Estimate'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df26['Functional_Class'],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#JAG1 supplement, added columns for aa_alt and aa_ref and functional class (according to author encoded functional class)
df27= pd.read_excel("~/Downloads/Pillar_project_data_files/JAG1_Gilbert_2024.xlsx", header = 1)
dc_27 = {"x": "~/Downloads/Pillar_project_data_files/JAG1_Gilbert_2024.xlsx", 
               "y": 1,
               "Dataset" :'JAG1_Gilbert_2024', 
               "Gene" : "JAG1", 
               "HGNC_id" : 6188, 
               "Chrom" : 20, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : df27["pos"], 
               "ref_allele": df27["Ref_Allele"], 
               "alt_allele": df27["Alt_Allele"], 
               "auth_transcript_id" : "NM_000214.3", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": df27["AA_Position"],
               "aa_ref": df27["aa_ref"],
               "aa_alt": df27["aa_alt"],
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": df27["Consequence"],
               "auth_reported_score": df27['meanAcrossReps'],
               "auth_reported_rep_score": df27[['VarScore_1', 'VarScore_2','VarScore_3','VarScore_4','VarScore_5',
                                               'VarScore_6','VarScore_7']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": df27['func_class'],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'Yes',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#BRCA2_Sahu_2023 supplement, new column added for hgvs_c without transcript annotation, "Intronic" removed from hgvs.p column
df28= pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA2_Sahu_2023_exon13.xlsx", header = 0)
dc_28 = {"x": "~/Downloads/Pillar_project_data_files/BRCA2_Sahu_2023_exon13.xlsx", 
               "y": 0,
               "Dataset" :'BRCA2_Sahu_2023_exon13_SGE', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df28["hgvs_c"],
               "hgvs_p" : df28["p.Nomenclature"],
               "consequence": np.nan,
               "auth_reported_score": df28['Function score DMSO'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": "nucleotide",
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

#BRCA2_Sahu_2023 supplement, new column added for hgvs_c without transcript annotation,"Intronic" removed from hgvs.p column
df29= pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA2_Sahu_2023_exon13.xlsx", header = 0)
dc_29 = {"x": "~/Downloads/Pillar_project_data_files/BRCA2_Sahu_2023_exon13.xlsx", 
               "y": 0,
               "Dataset" :'BRCA2_Sahu_2023_exon13_Cisplatin', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df29["hgvs_c"],
               "hgvs_p" : df29["p.Nomenclature"],
               "consequence": np.nan,
               "auth_reported_score": df29['Function score cisplatin'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

#BRCA2_Sahu_2023 supplement, new column added for hgvs_c without transcript annotation,"Intronic" removed from hgvs.p column
df30= pd.read_excel("~/Downloads/Pillar_project_data_files/BRCA2_Sahu_2023_exon13.xlsx", header = 0)
dc_30 = {"x": "~/Downloads/Pillar_project_data_files/BRCA2_Sahu_2023_exon13.xlsx", 
               "y": 0,
               "Dataset" :'BRCA2_Sahu_2023_exon13_Olaparib', 
               "Gene" : "BRCA2", 
               "HGNC_id" : 1101, 
               "Chrom" : 13, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000059.4", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": df30["hgvs_c"],
               "hgvs_p" : df30["p.Nomenclature"],
               "consequence": np.nan,
               "auth_reported_score": df30['Function score olaparib'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'Yes'}

#assertions according to authors added as a new column in the excel sheet 
df31= pd.read_excel("~/Downloads/Pillar_project_data_files/SCN5A_Ma_2024_current_density.xlsx", header = 0)
dc_31 = {"x": "~/Downloads/Pillar_project_data_files/SCN5A_Ma_2024_current_density.xlsx", 
               "y": 0,
               "Dataset" :'SCN5A_Ma_2024_current_density', 
               "Gene" : "SCN5A", 
               "HGNC_id" : 10593, 
               "Chrom" : 3, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : "NM_000335.5", 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df31["Cell line"],
               "consequence": np.nan,
               "auth_reported_score": df31['CD sqrtNORM(-120mV) Mean'],
               "auth_reported_rep_score": np.nan,
               "auth_reported_func_class": df31['class'],
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#TSC2 library 1 (tuberin domain), unpublished IGVF (Fowler lab)
df32 = pd.read_excel("~/Downloads/Pillar_project_data_files/TSC2_tuberin_unpublished.xlsx", header = 0)
dc_32 = {"x": "~/Downloads/Pillar_project_data_files/TSC2_tuberin_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'TSC2_tuberin_unpublished', 
               "Gene" : "TSC2", 
               "HGNC_id" : 12363, 
               "Chrom" : 16, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df32["aaChanges"],
               "consequence": np.nan,
               "auth_reported_score": df32['average'],
               "auth_reported_rep_score": df32[['abrep2', 'abrep3']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#TSC2 library 2 (rap gap domain), unpublished IGVF (Fowler lab)
df33 = pd.read_excel("~/Downloads/Pillar_project_data_files/TSC2_rapgap_unpublished.xlsx", header = 0)
dc_33 = {"x": "~/Downloads/Pillar_project_data_files/TSC2_rapgap_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'TSC2_rapgap_unpublished', 
               "Gene" : "TSC2", 
               "HGNC_id" : 12363, 
               "Chrom" : 16, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df33["aaChanges"],
               "consequence": np.nan,
               "auth_reported_score": df33['average'],
               "auth_reported_rep_score": df33[['abrep1','abrep2', 'abrep3']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#G6PD unppublished (Fowler lab)
df34 = pd.read_excel("~/Downloads/Pillar_project_data_files/G6PD_unpublished.xlsx", header = 0)
dc_34 = {"x": "~/Downloads/Pillar_project_data_files/G6PD_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'G6PD_unpublished', 
               "Gene" : "G6PD", 
               "HGNC_id" : 4057, 
               "Chrom" : 'X', 
               "hg19_pos" : np.nan, 
               "hg38_pos" : np.nan, 
               "ref_allele": np.nan, 
               "alt_allele": np.nan, 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : df34["aaChanges"],
               "consequence": np.nan,
               "auth_reported_score": df34['average'],
               "auth_reported_rep_score": df34[['abrep1','abrep2', 'abrep3']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'No',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'aa',
               "hgvs_p_conversion": 'Yes',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'Yes',
               "transcript_from_hgvs_c": 'No'}

#BARD1_unpublished (BBI)
df35 = pd.read_excel("~/Downloads/Pillar_project_data_files/BARD1_unpublished.xlsx", header = 0)
dc_35 = {"x": "~/Downloads/Pillar_project_data_files/BARD1_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'BARD1_unpublished', 
               "Gene" : "BARD1", 
               "HGNC_id" : 952, 
               "Chrom" : 2, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : df35['pos'], 
               "ref_allele": df35['ref'], 
               "alt_allele": df35['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df35['snv_score'],
               "auth_reported_rep_score": df35[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}
#PALB2_unpublished (BBI)
df36 = pd.read_excel("~/Downloads/Pillar_project_data_files/PALB2_unpublished.xlsx", header = 0)
dc_36 = {"x": "~/Downloads/Pillar_project_data_files/PALB2_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'PALB2_unpublished', 
               "Gene" : "PALB2", 
               "HGNC_id" : 26144, 
               "Chrom" : 16, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : df36['pos'], 
               "ref_allele": df36['ref'], 
               "alt_allele": df36['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df36['snv_score'],
               "auth_reported_rep_score": df36[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#XRCC2_unpublished (BBI)
df37 = pd.read_excel("~/Downloads/Pillar_project_data_files/XRCC2_unpublished.xlsx", header = 0)
dc_37 = {"x": "~/Downloads/Pillar_project_data_files/XRCC2_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'XRCC2_unpublished', 
               "Gene" : "XRCC2", 
               "HGNC_id" : 12829, 
               "Chrom" : 7, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : df37['pos'], 
               "ref_allele": df37['ref'], 
               "alt_allele": df37['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df37['snv_score'],
               "auth_reported_rep_score": df37[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#SFPQ_unpublished (BBI)
df38 = pd.read_excel("~/Downloads/Pillar_project_data_files/SFPQ_unpublished.xlsx", header = 0)
dc_38 = {"x": "~/Downloads/Pillar_project_data_files/SFPQ_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'SFPQ_unpublished', 
               "Gene" : "SFPQ", 
               "HGNC_id" : 10774, 
               "Chrom" : 1, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : df38['pos'], 
               "ref_allele": df38['ref'], 
               "alt_allele": df38['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df38['snv_score'],
               "auth_reported_rep_score": df38[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}

#NBN_unpublished (BBI)
df39 = pd.read_excel("~/Downloads/Pillar_project_data_files/NBN_unpublished.xlsx", header = 0)
dc_39 = {"x": "~/Downloads/Pillar_project_data_files/NBN_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'NBN_unpublished', 
               "Gene" : "NBN", 
               "HGNC_id" : 7652, 
               "Chrom" : 8, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : df39['pos'], 
               "ref_allele": df39['ref'], 
               "alt_allele": df39['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df39['snv_score'],
               "auth_reported_rep_score": df39[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}
#RAD51D_unpublished (BBI)
df40 = pd.read_excel("~/Downloads/Pillar_project_data_files/RAD51D_unpublished.xlsx", header = 0)
dc_40 = {"x": "~/Downloads/Pillar_project_data_files/RAD51D_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'RAD51D_unpublished', 
               "Gene" : "RAD51D", 
               "HGNC_id" : 9823, 
               "Chrom" : 17, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : df40['pos'], 
               "ref_allele": df40['ref'], 
               "alt_allele": df40['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df40['snv_score'],
               "auth_reported_rep_score": df40[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}
#CTCF_unpublished (BBI)
df41 = pd.read_excel("~/Downloads/Pillar_project_data_files/CTCF_unpublished.xlsx", header = 0)
dc_41 = {"x": "~/Downloads/Pillar_project_data_files/CTCF_unpublished.xlsx", 
               "y": 0,
               "Dataset" :'CTCF_unpublished', 
               "Gene" : "CTCF", 
               "HGNC_id" : 13723, 
               "Chrom" : 16, 
               "hg19_pos" : np.nan, 
               "hg38_pos" : df41['pos'], 
               "ref_allele": df41['ref'], 
               "alt_allele": df41['allele'], 
               "auth_transcript_id" : np.nan, 
               "transcript_pos" : np.nan,
               "transcript_ref" : np.nan, 
               "transcript_alt" : np.nan,
               "aa_pos": np.nan,
               "aa_ref": np.nan,
               "aa_alt": np.nan,
               "hgvs_c": np.nan,
               "hgvs_p" : np.nan,
               "consequence": np.nan,
               "auth_reported_score": df41['snv_score'],
               "auth_reported_rep_score": df41[['R1_score','R2_score', 'R3_score']].fillna('').astype(str).agg(';'.join, axis=1),
               "auth_reported_func_class": np.nan,
               "auth_reported_normal_min": np.nan,
               "auth_reported_normal_max": np.nan,
               "auth_reported_abnormal_min": np.nan,
               "auth_reported_abnormal_max":np.nan,
               "splice_measure": 'Yes',
               "gnomad_MAF": np.nan,
               "clinvar_sig": np.nan,
               "clinvar_star": np.nan, 
               "clinvar_date_last_reviewed": np.nan,
               "nucleotide_or_aa": 'nucleotide',
               "hgvs_p_conversion": 'No',
               "hgvs_from_aa": 'No',
               "aa_from_hgvs": 'No',
               "transcript_from_hgvs_c": 'No'}
#BAP1_Waters_2024
# df42 = pd.read_excel("~/Downloads/Pillar_project_data_files/BAP1_Waters_2024.xlsx", header = 0)
# dc_42 = {"x": "~/Downloads/Pillar_project_data_files/BAP1_Waters_2024.xlsx", 
#                "y": 2,
#                "Dataset" :'BAP1_Waters_2024', 
#                "Gene" : "BAP1", 
#                "HGNC_id" : 950, 
#                "Chrom" : 3, 
#                "hg19_pos" : np.nan, 
#                "hg38_pos" : df42['pos'], 
#                "ref_allele": df42['ref'], 
#                "alt_allele": df42['alt'], 
#                "auth_transcript_id" : 'ENST00000460680.6', 
#                "transcript_pos" : df42['CDS_position'],
#                "transcript_ref" : df42['ref'], 
#                "transcript_alt" : df42['alt'],
#                "aa_pos": df42['protein_position'],
#                "aa_ref": df42['ref_aa'],
#                "aa_alt": df42['alt_aa'],
#                "hgvs_c": df42['HGVSc'],
#                "hgvs_p" : df42['HGVSp'],
#                "consequence": df42['vep_consequence'],
#                "auth_reported_score": df42['functional_score'],
#                "auth_reported_rep_score": np.nan,
#                "auth_reported_func_class": df42['functional_classification'],
#                "auth_reported_normal_min": np.nan,
#                "auth_reported_normal_max": np.nan,
#                "auth_reported_abnormal_min": np.nan,
#                "auth_reported_abnormal_max":np.nan,
#                "splice_measure": 'Yes',
#                "gnomad_MAF": np.nan,
#                "clinvar_sig": np.nan,
#                "clinvar_star": np.nan, 
#                "clinvar_date_last_reviewed": np.nan,
#                "nucleotide_or_aa": 'nucleotide',
#                "hgvs_p_conversion": 'No',
#                "hgvs_from_aa": 'No',
#                "aa_from_hgvs": 'No',
#                "transcript_from_hgvs_c": 'No'}

dc_list = [dc_1, dc_2, dc_3, dc_4,dc_5,dc_6,dc_7,dc_8,dc_9,dc_10,dc_11,dc_12,dc_13,dc_14,dc_15,
           dc_16,dc_17,dc_18,dc_19,dc_20,dc_21,dc_22,dc_23, dc_24,dc_25,dc_26,dc_27,dc_28,dc_29,dc_30,dc_31,
          dc_31,dc_32,dc_33,dc_34,dc_35,dc_36,dc_37,dc_38,dc_39,dc_40,dc_41]

final_combined_df = data_harmonization_loop(dc_list)

final_combined_df.to_csv('~/Downloads/pillar_data_combined_df.csv', index=False)

FileNotFoundError: [Errno 2] No such file or directory: '/net/bbi/vol1/home/mtejura/Downloads/Pillar_project_data_files/BRCA1_Findlay_2018.xlsx'

In [2]:
import pandas as pd

p_data = pd.read_csv("~/pillar_data_combined_df.csv")

condition = (p_data['Dataset'] == 'BRCA2_Hu_2024') | (p_data['Dataset'] == 'BRCA2_Sahu_2023_exon13_SGE') | (p_data['Dataset'] == 'BRCA2_Sahu_2023_exon13_Olaparib') | (p_data['Dataset'] == 'BRCA2_Sahu_2023_exon13_Cisplatin')
    
#BRCA2 is on the positive strand so transcript ref and alt would be the same as genome ref and alt
p_data.loc[condition,'ref_allele'] = p_data.loc[condition,'transcript_ref']

p_data.loc[condition,'alt_allele'] = p_data.loc[condition,'transcript_alt']


curation_data = pd.read_csv("~/MAVE_curation.csv", header = 1)

df_merge = pd.merge(p_data, curation_data[['Dataset_tag', 'MaveDB URN', 'Ensembl_transript_ID', 
                                           "Ref_seq_transcript_ID","Model_system","Assay_type",
                                          "Phenotype_measured","Phenotype_detail","IGVF_produced"]], 
                                           left_on='Dataset', right_on = "Dataset_tag", how='left')

df_merge.to_csv("~/pillar_data_with_curation.csv", index = False)

/tmp/6075162.1.fowler-login.q/ipykernel_4187203/906618730.py:3: DtypeWarning: Columns (3,6,7,8,9,10,11,12,13,14,15,16,17,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  p_data = pd.read_csv("~/pillar_data_combined_df.csv")


In [57]:
import pandas as pd
import re

# Load your initial data
pd_1 = pd.read_csv("~/pillar_data_with_curation.csv", header=0)

# Define the amino acid to codon dictionary
amino_acid_to_codon = {
    'F': ['TTT', 'TTC'],
    'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
    'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
    'Y': ['TAT', 'TAC'],
    'Ter': ['TAA', 'TAG', 'TGA'],
    '*': ['TAA', 'TAG', 'TGA'],
    'C': ['TGT', 'TGC'],
    'W': ['TGG'],
    'P': ['CCT', 'CCC', 'CCA', 'CCG'],
    'H': ['CAT', 'CAC'],
    'Q': ['CAA', 'CAG'],
    'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
    'I': ['ATT', 'ATC', 'ATA'],
    'M': ['ATG'],
    'T': ['ACT', 'ACC', 'ACA', 'ACG'],
    'N': ['AAT', 'AAC'],
    'K': ['AAA', 'AAG'],
    'V': ['GTT', 'GTC', 'GTA', 'GTG'],
    'A': ['GCT', 'GCC', 'GCA', 'GCG'],
    'D': ['GAT', 'GAC'],
    'E': ['GAA', 'GAG'],
    'G': ['GGT', 'GGC', 'GGA', 'GGG']
}

# Specify datasets that need the chromosome-based key generation approach
datasets_with_chromosome_key = {"BARD1_unpublished", "CTCF_unpublished", 'NBN_unpublished','BARD1_unpublished', 
                                'PALB2_unpublished','RAD51D_unpublished','SFPQ_unpublished','XRCC2_unpublished','BRCA1_Findlay_2018'}  # Replace with actual dataset names

expanded_rows = []

for i, row in pd_1.iterrows():
    dataset = row['Dataset']
    
    if dataset in datasets_with_chromosome_key:
        # Generate chromosome-based key
        chrom = row['Chrom']
        pos = row['hg38_pos']
        ref = row['ref_allele']
        alt = row['alt_allele']
        
        try:
            chrom = int(chrom)
            pos = int(pos)
            chrom_str = f"{chrom:02}" if chrom < 10 else str(chrom)
            key = f"NC_0000{chrom_str}:g.{pos}{ref}>{alt}"
            
            expanded_row = row.to_dict()  
            expanded_row['key'] = key     
            expanded_rows.append(expanded_row)

        except KeyError as e:
            # print(f"KeyError for row {i} with value {value}: {e}")
            # Append the row with a placeholder key
            expanded_row = row.to_dict()
            expanded_row['key'] = 'KeyError'
            expanded_rows.append(expanded_row)
        except ValueError as e:
            # print(f"ValueError for row {i} with position {pos}: {e}")
            # Append the row with a placeholder key
            expanded_row = row.to_dict()
            expanded_row['key'] = 'ValueError'
            expanded_rows.append(expanded_row)
        except Exception as e:
            # print(f"Unexpected error for row {i}: {e}")
            # Append the row with a generic error key
            expanded_row = row.to_dict()
            expanded_row['key'] = 'Error'
            expanded_rows.append(expanded_row)
        

    else:
        # Generate transcript-based key
        value = row['aa_alt']
        pos = row['aa_pos']
        ID = row['Ensembl_transript_ID']
        
        try:
            if pd.notna(ID) and isinstance(ID, str) and value in amino_acid_to_codon:
                transcript = re.sub(r"(ENST\d+)\.\d+", r"\1", ID)
                codons = amino_acid_to_codon[value]
                nucleotide_pos = (int(float(pos)) * 3) - 2
                
                for codon in codons:
                    key = f"{transcript}:c.{int(nucleotide_pos)}_{int(nucleotide_pos+2)}delins{codon}"
                    
                    expanded_row = row.to_dict()  
                    expanded_row['key'] = key     
                    expanded_rows.append(expanded_row)
        except KeyError as e:
            print(f"KeyError for row {i} with value {value}: {e}")
            continue
        except ValueError as e:
            print(f"ValueError for row {i} with position {pos}: {e}")
            continue
        

# Convert expanded rows into a new DataFrame with the key
expanded_df = pd.DataFrame(expanded_rows)

# Save to CSV
expanded_df.to_csv("~/pillar_data_with_curation_with_key.csv", index=False)

/tmp/6075162.1.fowler-login.q/ipykernel_4187203/672000500.py:5: DtypeWarning: Columns (3,6,7,8,9,10,11,12,13,14,15,16,17,19,20,32) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_1 = pd.read_csv("~/pillar_data_with_curation.csv", header=0)


ValueError for row 0 with position nan: cannot convert float NaN to integer
ValueError for row 1 with position nan: cannot convert float NaN to integer
ValueError for row 2 with position nan: cannot convert float NaN to integer
ValueError for row 3 with position nan: cannot convert float NaN to integer
ValueError for row 4 with position nan: cannot convert float NaN to integer
ValueError for row 5 with position nan: cannot convert float NaN to integer
ValueError for row 6 with position nan: cannot convert float NaN to integer
ValueError for row 7 with position nan: cannot convert float NaN to integer
ValueError for row 8 with position nan: cannot convert float NaN to integer
ValueError for row 9 with position nan: cannot convert float NaN to integer
ValueError for row 10 with position nan: cannot convert float NaN to integer
ValueError for row 11 with position nan: cannot convert float NaN to integer
ValueError for row 12 with position nan: cannot convert float NaN to integer
ValueErro

In [59]:
import pandas as pd

pd_1_key = pd.read_csv("~/pillar_data_with_curation_with_key.csv")

pd_2_key = pd.read_csv("~/VEP_output_v2_1.csv")

pd_2_key = pd_2_key.applymap(lambda x: x.strip() if isinstance(x, str) else x)

pd_2_key = pd_2_key.apply(pd.to_numeric, errors='ignore')

pd_3 = pd_2_key.drop_duplicates()

pd_1_filtered = pd_1_key[pd_1_key['nucleotide_or_aa'] == 'aa']

pd_merged = pd.merge(pd_1_filtered, pd_3[["Location","Allele","Consequence","HGVSc","HGVSp",
                                          "cDNA_position","CDS_position","Protein_position","Amino_acids",
                                          "Codons","REF_ALLELE","Feature","#Uploaded_variation","UPLOADED_ALLELE",
                                          "STRAND"]], 
                                          left_on=["key",'Ensembl_transript_ID'],
                                          right_on = ["#Uploaded_variation","Feature"],how='left')

/tmp/6075162.1.fowler-login.q/ipykernel_4187203/3950684447.py:3: DtypeWarning: Columns (3,6,7,8,9,10,11,13,14,15,16,17,19,20,32,35,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_1_key = pd.read_csv("~/pillar_data_with_curation_with_key.csv")
/tmp/6075162.1.fowler-login.q/ipykernel_4187203/3950684447.py:5: DtypeWarning: Columns (16,23,30,34) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_2_key = pd.read_csv("~/VEP_output_v2_1.csv")
/tmp/6075162.1.fowler-login.q/ipykernel_4187203/3950684447.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pd_2_key = pd_2_key.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/6075162.1.fowler-login.q/ipykernel_4187203/3950684447.py:9: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  pd_2_key = pd_2_key.apply(pd.to_num

In [60]:
pd_1_key

,Dataset,Gene,HGNC_id,Chrom,hg19_pos,hg38_pos,ref_allele,alt_allele,auth_transcript_id,transcript_pos,transcript_ref,transcript_alt,aa_pos,aa_ref,aa_alt,hgvs_c,hgvs_p,consequence,auth_reported_score,auth_reported_rep_score,auth_reported_func_class,auth_reported_normal_min,auth_reported_normal_max,auth_reported_abnormal_min,auth_reported_abnormal_max,splice_measure,gnomad_MAF,clinvar_sig,clinvar_star,clinvar_date_last_reviewed,nucleotide_or_aa,Dataset_tag,MaveDB URN,Ensembl_transript_ID,Ref_seq_transcript_ID,Model_system,Assay_type,Phenotype_measured,Phenotype_detail,IGVF_produced,key
0,BRCA1_Findlay_2018,BRCA1,1100,17,41276135.0,NaN,T,G,NM_007294.3,-19-3,A,C,NaN,NaN,NaN,c.-19-3A>C,NaN,Splice region,-0.372611,-0.568675758504309;-0.176545248348852,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,NaN,NaN,NaN,nucleotide,BRCA1_Findlay_2018,urn:mavedb:00000097-0-1,ENST00000357654.9,NM_007294.3,immortalized human cells,Cell Viability,Cell Survival,Overall function,No,ValueError
1,BRCA1_Findlay_2018,BRCA1,1100,17,41276135.0,NaN,T,C,NM_007294.3,-19-3,A,G,NaN,NaN,NaN,c.-19-3A>G,NaN,Splice region,-0.045313,-0.332667114078832;0.242040175629815,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,NaN,NaN,NaN,nucleotide,BRCA1_Findlay_2018,urn:mavedb:00000097-0-1,ENST00000357654.9,NM_007294.3,immortalized human cells,Cell Viability,Cell Survival,Overall function,No,ValueError
2,BRCA1_Findlay_2018,BRCA1,1100,17,41276135.0,NaN,T,A,NM_007294.3,-19-3,A,T,NaN,NaN,NaN,c.-19-3A>T,NaN,Splice region,-0.108254,-0.440230467981572;0.223722191334123,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,NaN,NaN,NaN,nucleotide,BRCA1_Findlay_2018,urn:mavedb:00000097-0-1,ENST00000357654.9,NM_007294.3,immortalized human cells,Cell Viability,Cell Survival,Overall function,No,ValueError
3,BRCA1_Findlay_2018,BRCA1,1100,17,41276134.0,NaN,T,G,NM_007294.3,-19-2,A,C,NaN,NaN,NaN,c.-19-2A>C,NaN,Canonical splice,-0.277963,-0.41057727764695;-0.145348986773218,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,NaN,NaN,NaN,nucleotide,BRCA1_Findlay_2018,urn:mavedb:00000097-0-1,ENST00000357654.9,NM_007294.3,immortalized human cells,Cell Viability,Cell Survival,Overall function,No,ValueError
4,BRCA1_Findlay_2018,BRCA1,1100,17,41276134.0,NaN,T,C,NM_007294.3,-19-2,A,G,NaN,NaN,NaN,c.-19-2A>G,NaN,Canonical splice,-0.388414,-0.6350191556350679;-0.141808664620659,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,NaN,NaN,NaN,nucleotide,BRCA1_Findlay_2018,urn:mavedb:00000097-0-1,ENST00000357654.9,NM_007294.3,immortalized human cells,Cell Viability,Cell Survival,Overall function,No,ValueError
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428994,CTCF_unpublished,CTCF,13723,16,NaN,67626643.0,C,G,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.959996,-0.6541320779744095;0.1299313120361334;0.23398...,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,nucleotide,CTCF_unpublished,NaN,ENST00000264010.10,NM_006565.4,immortalized human cells,Cell viability,Cell survival,overall function,Yes,NC_000016:g.67626643C>G
428995,CTCF_unpublished,CTCF,13723,16,NaN,67626643.0,C,T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.901529,-0.157539930598156;-0.0646627700816877;0.28777...,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,nucleotide,CTCF_unpublished,NaN,ENST00000264010.10,NM_006565.4,immortalized human cells,Cell viability,Cell survival,overall function,Yes,NC_000016:g.67626643C>T
428996,CTCF_unpublished,CTCF,13723,16,NaN,67626644.0,A,C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.040071,0.3758153232205121;0.6194697060103307;0.396447...,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,nucleotide,CTCF_unpublished,NaN,ENST00000264010.10,NM_006565.4,immortalized human cells,Cell viability,Cell survival,overall function,Yes,NC_000016:g.67626644A>C
428997,CTCF_unpublished,CTCF,13723,16,NaN,67626644.0,A,G,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.942589,-0.3943383820371419;0.0719964942613049;0.60524...,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,nucleotide,CTCF_unpublished,NaN,ENST0

In [61]:
pd_merged = pd_merged.applymap(lambda x: x.strip() if isinstance(x, str) else x)

pd_merged = pd_merged.apply(pd.to_numeric, errors='ignore')

pd_merged_clean = pd_merged.drop_duplicates()

print(len(pd_merged))

print(len(pd_merged_clean))

/tmp/6075162.1.fowler-login.q/ipykernel_4187203/4044167027.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pd_merged = pd_merged.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/6075162.1.fowler-login.q/ipykernel_4187203/4044167027.py:3: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  pd_merged = pd_merged.apply(pd.to_numeric, errors='ignore')


598778
592773


In [62]:
import re

nuc_dic = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', '-':'-'}

def translate_sequence(sequence):
    return ''.join(nuc_dic.get(nuc, nuc) for nuc in sequence)[::-1]

def process_alleles(row):
    if pd.notna(row['UPLOADED_ALLELE']):
        ref, alt = str(row['UPLOADED_ALLELE']).split("/")
        if row['STRAND'] == -1:
            return translate_sequence(ref), translate_sequence(alt)
        return ref, alt
    return row['UPLOADED_ALLELE'], row['UPLOADED_ALLELE']

pd_merged_clean[['ref_allele', 'alt_allele']] = pd_merged_clean.apply(process_alleles, axis=1, result_type="expand")


def extract_hg38_position(value):
    if pd.notna(value):
        chrom, pos = str(value).split(":")
        start, end = pos.split("-")
        return start, end
    return value, value

pd_merged_clean[['hg38_pos', 'hg38_end']] = pd_merged_clean['Location'].apply(extract_hg38_position).apply(pd.Series)

pd_merged_clean['hgvs_c'] = pd_merged_clean['HGVSc'].apply(lambda x: x.split(":")[1] if pd.notna(x) and ":" in x else x)

pd_merged_clean['transcript_pos'] = pd_merged_clean['CDS_position']
pd_merged_clean['consequence'] = pd_merged_clean['Consequence']

pd_merged_clean.head()

/tmp/6075162.1.fowler-login.q/ipykernel_4187203/3404314128.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_merged_clean[['ref_allele', 'alt_allele']] = pd_merged_clean.apply(process_alleles, axis=1, result_type="expand")
/tmp/6075162.1.fowler-login.q/ipykernel_4187203/3404314128.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pd_merged_clean[['hg38_pos', 'hg38_end']] = pd_merged_clean['Location'].apply(extract_hg38_position).apply(pd.Series)
/tmp/6075162.1.fowler-login.q/ipykernel_4187203/34043

,Dataset,Gene,HGNC_id,Chrom,hg19_pos,hg38_pos,ref_allele,alt_allele,auth_transcript_id,transcript_pos,transcript_ref,transcript_alt,aa_pos,aa_ref,aa_alt,hgvs_c,hgvs_p,consequence,auth_reported_score,auth_reported_rep_score,auth_reported_func_class,auth_reported_normal_min,auth_reported_normal_max,auth_reported_abnormal_min,auth_reported_abnormal_max,splice_measure,gnomad_MAF,clinvar_sig,clinvar_star,clinvar_date_last_reviewed,nucleotide_or_aa,Dataset_tag,MaveDB URN,Ensembl_transript_ID,Ref_seq_transcript_ID,Model_system,Assay_type,Phenotype_measured,Phenotype_detail,IGVF_produced,key,Location,Allele,Consequence,HGVSc,HGVSp,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,REF_ALLELE,Feature,#Uploaded_variation,UPLOADED_ALLELE,STRAND,hg38_end
0,BRCA1_Adamovich_2022_HDR,BRCA1,1100,17,NaN,43071184,AG,TT,NaN,4729-4730,NaN,NaN,1577.0,S,*,c.4729_4730delinsAA,p.Ser1577Ter,missense_variant,-1.068371,-1.241803672;-1.550598196;-0.495599456;-0.9854...,NaN,-0.233,1.357,-3.198,-0.562,No,NaN,NaN,NaN,NaN,aa,BRCA1_Adamovich_2022_HDR,urn:mavedb:00001208-a-2,ENST00000357654.9,NM_007294.4,immortalized human cells,Reporter,Fluorescence,Homology Direted Repair,No,ENST00000357654:c.4729_4731delinsTAA,17:43071184-43071185,TT,missense_variant,ENST00000357654.9:c.4729_4730delinsAA,ENSP00000350283.3:p.Ser1577Asn,4842-4843,4729-4730,1577,S/N,TCt/AAt,CT,ENST00000357654.9,ENST00000357654:c.4729_4731delinsTAA,CT/AA,-1.0,43071185
1,BRCA1_Adamovich_2022_HDR,BRCA1,1100,17,NaN,43071184,AG,CT,NaN,4729-4730,NaN,NaN,1577.0,S,*,c.4729_4730delinsAG,p.Ser1577Ter,synonymous_variant,-1.068371,-1.241803672;-1.550598196;-0.495599456;-0.9854...,NaN,-0.233,1.357,-3.198,-0.562,No,NaN,NaN,NaN,NaN,aa,BRCA1_Adamovich_2022_HDR,urn:mavedb:00001208-a-2,ENST00000357654.9,NM_007294.4,immortalized human cells,Reporter,Fluorescence,Homology Direted Repair,No,ENST00000357654:c.4729_4731delinsTAG,17:43071184-43071185,CT,synonymous_variant,ENST00000357654.9:c.4729_4730delinsAG,ENSP00000350283.3:p.Ser1577%3D,4842-4843,4729-4730,1577,S,TCt/AGt,CT,ENST00000357654.9,ENST00000357654:c.4729_4731delinsTAG,CT/AG,-1.0,43071185
2,BRCA1_Adamovich_2022_HDR,BRCA1,1100,17,NaN,43071184,AG,TC,NaN,4729-4730,NaN,NaN,1577.0,S,*,c.4729_4730inv,p.Ser1577Ter,missense_variant,-1.068371,-1.241803672;-1.550598196;-0.495599456;-0.9854...,NaN,-0.233,1.357,-3.198,-0.562,No,NaN,NaN,NaN,NaN,aa,BRCA1_Adamovich_2022_HDR,urn:mavedb:00001208-a-2,ENST00000357654.9,NM_007294.4,immortalized human cells,Reporter,Fluorescence,Homology Direted Repair,No,ENST00000357654:c.4729_4731delinsTGA,17:43071184-43071185,TC,missense_variant,ENST00000357654.9:c.4729_4730inv,ENSP00000350283.3:p.Ser1577Asp,4842-4843,4729-4730,1577,S/D,TCt/GAt,CT,ENST00000357654.9,ENST00000357654:c.4729_4731delinsTGA,CT/GA,-1.0,43071185
3,BRCA1_Adamovich_2022_HDR,BRCA1,1100,17,NaN,43071183,A,C,NaN,4731,NaN,NaN,1577.0,S,A,c.4731T>G,p.Ser1577Ala,synonymous_variant,-0.232972,-0.331299807;-0.402713276;-0.536221202;0.33834...,NaN,-0.233,1.357,-3.198,-0.562,No,NaN,NaN,NaN,NaN,aa,BRCA1_Adamovich_2022_HDR,urn:mavedb:00001208-a-2,ENST00000357654.9,NM_007294.4,immortalized human cells,Reporter,Fluorescence,Homology Direted Repair,No,ENST00000357654:c.4729_4731delinsGCT,17:43071183-43071183,C,synonymous_variant,ENST00000357654.9:c.4731T>G,ENSP00000350283.3:p.Ser1577%3D,4844,4731,1577,S,tcT/tcG,T,ENST00000357654.9,ENST00000357654:c.4729_4731delinsGCT,T/G,-1.0,43071183
4,BRCA1_Adamovich_2022_HDR,BRCA1,1100,17,NaN,43071183,AGA,GGC,NaN,4729-4731,NaN,NaN,1577.0,S,A,c.4729_4731delinsGCC,p.Ser1577Ala,missense_variant,-0.232972,-0.331299807;-0.402713276;-0.536221202;0.33834...,NaN,-0.233,1.357,-3.198,-0.562,No,NaN,NaN,NaN,NaN,aa,BRCA1_Adamovich_2022_HDR,urn:mavedb:00001208-a-2,ENST00000357654.9,NM_007294.4,immortalized human cells,Reporter,Fluorescence,Homology Direted Repair,No,ENST00000357654:c.4729_4731delinsGCC,17:43071183-43071185,GGC,missense_variant,ENST00000357654.9:c.4729_4731delinsGCC,ENSP00000350283.3:p.Ser1577Ala,4842-4844,4729-4731,1577,S/A,TCT/GCC,

In [21]:
pd_2_key = pd.read_csv("~/VEP_output_v2_1.csv")


genes_to_filter = ['BRCA1','BRCA2','BARD1','PALB2','XRCC2','CTCF','NBN','RAD51D','SFPQ','VHL','JAG1']

# Query using .isin()
pd_2_key_filt = pd_2_key[pd_2_key['SYMBOL'].isin(genes_to_filter)]


def process_alleles_nucleotide(row):
    if pd.notna(row['UPLOADED_ALLELE']):
        ref, alt = str(row['UPLOADED_ALLELE']).split("/")
        return ref, alt
    return row['UPLOADED_ALLELE'], row['UPLOADED_ALLELE']

pd_2_key_filt[['ref_allele', 'alt_allele']] = pd_2_key_filt.apply(process_alleles_nucleotide, axis=1, result_type="expand")


def extract_hg38_position(value):
    if pd.notna(value):
        chrom, pos = str(value).split(":")
        start, end = pos.split("-")
        return start, end
    return value, value

pd_2_key_filt[['hg38_pos', 'hg38_end']] = pd_2_key_filt['Location'].apply(extract_hg38_position).apply(pd.Series)

def amino_acids_nucleotide(row):
    if pd.notna(row['Amino_acids']):
        amino_acids = str(row['Amino_acids']).split("/")
        if len(amino_acids) == 2:
            ref_aa, alt_aa = amino_acids
        else:
            ref_aa = alt_aa = amino_acids[0]
        return ref_aa, alt_aa
    return row['Amino_acids'], row['Amino_acids']


pd_2_key_filt[['ref_aa', 'alt_aa']] = pd_2_key_filt.apply(amino_acids_nucleotide, axis=1, result_type="expand")

pd_2_key_filt.to_csv("~/expanded_vep_data_v2.csv", index = False)


In [12]:
pd_2_key_filt.columns

Index(['#Uploaded_variation', 'Location', 'Allele', 'Consequence', 'IMPACT',
       'SYMBOL', 'Gene', 'Feature_type', 'Feature', 'BIOTYPE', 'EXON',
       'INTRON', 'HGVSc', 'HGVSp', 'cDNA_position', 'CDS_position',
       'Protein_position', 'Amino_acids', 'Codons', 'Existing_variation',
       'REF_ALLELE', 'UPLOADED_ALLELE', 'DISTANCE', 'STRAND', 'FLAGS',
       'SYMBOL_SOURCE', 'HGNC_ID', 'MANE', 'MANE_SELECT', 'MANE_PLUS_CLINICAL',
       'TSL', 'APPRIS', 'SIFT', 'PolyPhen', 'HGVS_OFFSET', 'AF', 'CLIN_SIG',
       'SOMATIC', 'PHENO', 'PUBMED', 'MOTIF_NAME', 'MOTIF_POS', 'HIGH_INF_POS',
       'MOTIF_SCORE_CHANGE', 'TRANSCRIPTION_FACTORS', 'ref_allele',
       'alt_allele', 'hg38_pos', 'hg38_end', 'ref_aa', 'alt_aa'],
      dtype='object')

In [63]:
pd_2_key_filt = pd.read_csv("~/expanded_vep_data_v2.csv")

final_df = pd.merge(pd_1_key, pd_merged_clean, on=['Dataset', 'key'], how='left', suffixes=('', '_drop'))

columns_to_update = ['hg38_pos', 'ref_allele', 'alt_allele','transcript_pos',
       'transcript_ref', 'transcript_alt','consequence']
columns_with_fallback = ['hg38_pos_drop', 'ref_allele_drop', 'alt_allele_drop',
                        'transcript_pos_drop', 'transcript_ref_drop',
                        'transcript_alt_drop','consequence_drop']

for x_col, y_col in zip(columns_to_update, columns_with_fallback):
    final_df[x_col] = final_df[x_col].fillna(final_df[y_col])  


final_df = final_df[[col for col in final_df.columns if not col.endswith('_drop')]]

#need to merge nucleotide right

final_df['hg38_pos'] = final_df['hg38_pos'].astype(float)

pd_2_key_filt['hg38_pos'] = pd_2_key_filt['hg38_pos'].astype(float)

pd_2_key_filt = pd_2_key_filt.drop_duplicates()

final_df_2 = pd.merge(final_df, pd_2_key_filt[["Location","Allele","Consequence","HGVSc","HGVSp",
                                          "cDNA_position","CDS_position","Protein_position","Amino_acids",
                                          "Codons","REF_ALLELE","Feature","#Uploaded_variation","UPLOADED_ALLELE",
                                          "STRAND",'ref_allele','alt_allele', 'hg38_pos', 'hg38_end', 'ref_aa', 'alt_aa']], 
                                          left_on=['Ensembl_transript_ID','ref_allele','alt_allele','hg38_pos'],
                                          right_on = ['Feature','ref_allele','alt_allele', 'hg38_pos'],how='left')


final_df_2['hgvs_c'] = final_df_2['HGVSc_y'].apply(lambda x: x.split(":")[1] if pd.notna(x) and ":" in x else x)

final_df_2['transcript_pos'] = final_df_2['transcript_pos'].fillna(final_df_2['CDS_position_y'])

final_df_2['consequence'] = final_df_2['consequence'].fillna(final_df_2['Consequence_y'])

final_df_2['aa_ref'] = final_df_2['aa_ref'].fillna(final_df_2['ref_aa'])

final_df_2['aa_alt'] = final_df_2['aa_alt'].fillna(final_df_2['alt_aa'])

final_df_2['aa_pos'] = final_df_2['aa_pos'].fillna(final_df_2['Protein_position_y'])

final_df_2['hg38_pos'] = final_df_2['hg38_pos'].fillna(final_df_2['hg38_pos'])

final_df_2['hg38_end'] = final_df_2['hg38_end_x'].fillna(final_df_2['hg38_end_y'])

final_df_2.drop(columns=['Dataset_tag','key', 'Location_x', 'Allele_x', 'Consequence_x',
       'HGVSc_x', 'HGVSp_x', 'cDNA_position_x', 'CDS_position_x',
       'Protein_position_x', 'Amino_acids_x', 'Codons_x', 'REF_ALLELE_x',
       'Feature_x', '#Uploaded_variation_x', 'UPLOADED_ALLELE_x', 'STRAND_x',
       'hg38_end_x', 'Location_y', 'Allele_y', 'Consequence_y', 'HGVSc_y',
       'HGVSp_y', 'cDNA_position_y', 'CDS_position_y', 'Protein_position_y',
       'Amino_acids_y', 'Codons_y', 'REF_ALLELE_y', 'Feature_y',
       '#Uploaded_variation_y', 'UPLOADED_ALLELE_y', 'STRAND_y', 'hg38_end_y',
       'ref_aa', 'alt_aa'], inplace=True)

new_column_order = ['Dataset', 'Gene', 'HGNC_id', 'Chrom', 'hg19_pos', 'hg38_pos','hg38_end',
       'ref_allele', 'alt_allele', 'auth_transcript_id', 'transcript_pos',
       'transcript_ref', 'transcript_alt', 'aa_pos', 'aa_ref', 'aa_alt',
       'hgvs_c', 'hgvs_p', 'consequence', 'auth_reported_score',
       'auth_reported_rep_score', 'auth_reported_func_class',
       'auth_reported_normal_min', 'auth_reported_normal_max',
       'auth_reported_abnormal_min', 'auth_reported_abnormal_max',
       'splice_measure', 'gnomad_MAF', 'clinvar_sig', 'clinvar_star',
       'clinvar_date_last_reviewed', 'nucleotide_or_aa', 'MaveDB URN',
       'Ensembl_transript_ID', 'Ref_seq_transcript_ID', 'Model_system',
       'Assay_type', 'Phenotype_measured', 'Phenotype_detail', 'IGVF_produced']

final_df_2 = final_df_2[new_column_order]

final_df_2 = final_df_2.rename(columns={'hg38_pos': 'hg38_start'})

/tmp/6075162.1.fowler-login.q/ipykernel_4187203/173956738.py:1: DtypeWarning: Columns (30,34) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_2_key_filt = pd.read_csv("~/expanded_vep_data_v2.csv")


In [64]:
final_df_2.to_csv("~/final_pillar_data_v2.csv", index = False)

In [109]:
import gzip

chromosome = []
position = []
ref_allele = []
alt_allele = []
Clnsig = []
clnvc = []
Gene = []
variant_type = []
temp_list = []
temp_list2 = []
final_list = []

def clinvar_38(file1):
    with gzip.open(file1, "rt") as my_file:
        for line in my_file:
            if line[0] != "#":
                reader = line.split("\t")
                chromosome.append(reader[0])
                position.append(reader[1])
                ref_allele.append(reader[3])
                alt_allele.append(reader[4])
                significance = str(reader[7]).find("CLNSIG=")
                var_type = str(reader[7]).find("CLNVC=")
                clnvcso = str(reader[7]).find("CLNVCSO")
                origin = str(reader[7]).find("ORIGIN")             
                gene = str(reader[7]).find("GENEINFO")
                mc = str(reader[7]).find("MC=")
                temp_list2.append(reader[7][int(mc): int(origin)])
                temp_list.append(reader[7][int(gene):int(mc)])
                Clnsig.append((reader[7][int(significance)+7:int(var_type)-1]))
                clnvc.append((reader[7][int(var_type)+6:int(clnvcso)-1]))

        for element in temp_list:
            colon = element.find(":")
            Gene.append(element[9:int(colon)])
                
        for element in temp_list2:
            dash = element.find("|")
            semi = element.find(";")
            variant_type.append(element[14:-1])

    
    for i in range(0,len(chromosome)):
            mini_list = []
            mini_list.append(chromosome[i])
            mini_list.append(position[i])
            mini_list.append(ref_allele[i])
            mini_list.append(alt_allele[i])
            mini_list.append(Clnsig[i])
            mini_list.append(clnvc[i])
            mini_list.append(Gene[i])
            mini_list.append(variant_type[i])
            final_list.append(mini_list)
   
    import csv
    
    import pandas as pd
    
    import os.path
    
    header = ["Chromosome","Genomic_coordinates","Ref_allele","Alt_allele","Significance","Variant","Gene","Variant_type"]

    save_path = "/Users/malvikatejura/Downloads"
    
    file_name2 = "clinvar_wg_10152024" + ".csv"
    
    saved = os.path.join(save_path, file_name2)


    with open(saved,"w",encoding = "UTF8", newline = '') as new_file:
        writer = csv.writer(new_file)
        writer.writerow(header)
        for element in final_list:
            writer.writerow(element)
            
            
    df1 = pd.read_csv(saved, sep = ",", header = 0, dtype = {"Chromosome": str , "Genomic_coordinates" : int})
    
    df1.to_csv(path_or_buf= file_name2)
            
clinvar_38("/Users/malvikatejura/Downloads/clinvar_10152024.vcf.gz")

In [65]:
import pandas as pd

gh = pd.read_csv("~/final_pillar_data_v2.csv")

hh = pd.read_csv("~/clinvar_wg_10152024.csv")

/tmp/6075162.1.fowler-login.q/ipykernel_4187203/1718976444.py:3: DtypeWarning: Columns (3,9,11,12,13,16,17,20,21,32,35,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  gh = pd.read_csv("~/final_pillar_data_v2.csv")
/tmp/6075162.1.fowler-login.q/ipykernel_4187203/1718976444.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  hh = pd.read_csv("~/clinvar_wg_10152024.csv")


In [66]:
hh['Genomic_coordinates'] = hh['Genomic_coordinates'].astype(float)

merged = pd.merge(gh, hh, left_on=['Gene', 'ref_allele','alt_allele','hg38_start'], right_on = ['Gene',"Ref_allele",
                                   "Alt_allele",'Genomic_coordinates'], how='left')

columns_to_update_1 = ['clinvar_sig']
columns_with_fallback_1 = ['Significance']

for x_col, y_col in zip(columns_to_update_1, columns_with_fallback_1):
    merged[x_col] = merged[x_col].fillna(merged[y_col])

merged.drop(columns=['Chromosome', 'Genomic_coordinates', 'Ref_allele',
       'Alt_allele', 'Variant', 'Variant_type','Unnamed: 0',"Significance"], inplace=True)

print(merged['clinvar_sig'])

merged.to_csv("~/pillar_data_clinvar38_test_annotated.csv", index = False)

0         NaN
1         NaN
2         NaN
3         NaN
4         NaN
         ... 
650743    NaN
650744    NaN
650745    NaN
650746    NaN
650747    NaN
Name: clinvar_sig, Length: 650748, dtype: object


In [204]:
import gzip

chromosome = []
position = []
ref_allele = []
alt_allele = []
Clnsig = []
clnvc = []
Gene = []
variant_type = []
temp_list = []
temp_list2 = []
final_list = []

def clinvar_38(file1):
    with gzip.open(file1, "rt") as my_file:
        for line in my_file:
            if line[0] != "#":
                reader = line.split("\t")
                chromosome.append(reader[0])
                position.append(reader[1])
                ref_allele.append(reader[3])
                alt_allele.append(reader[4])
                significance = str(reader[7]).find("CLNSIG=")
                var_type = str(reader[7]).find("CLNVC=")
                clnvcso = str(reader[7]).find("CLNVCSO")
                origin = str(reader[7]).find("ORIGIN")             
                gene = str(reader[7]).find("GENEINFO")
                mc = str(reader[7]).find("MC=")
                temp_list2.append(reader[7][int(mc): int(origin)])
                temp_list.append(reader[7][int(gene):int(mc)])
                Clnsig.append((reader[7][int(significance)+7:int(var_type)-1]))
                clnvc.append((reader[7][int(var_type)+6:int(clnvcso)-1]))

        for element in temp_list:
            colon = element.find(":")
            Gene.append(element[9:int(colon)])
                
        for element in temp_list2:
            dash = element.find("|")
            semi = element.find(";")
            variant_type.append(element[14:-1])

    
    for i in range(0,len(chromosome)):
            mini_list = []
            mini_list.append(chromosome[i])
            mini_list.append(position[i])
            mini_list.append(ref_allele[i])
            mini_list.append(alt_allele[i])
            mini_list.append(Clnsig[i])
            mini_list.append(clnvc[i])
            mini_list.append(Gene[i])
            mini_list.append(variant_type[i])
            final_list.append(mini_list)
   
    import csv
    
    import pandas as pd
    
    import os.path
    
    header = ["Chromosome","Genomic_coordinates","Ref_allele","Alt_allele","Significance","Variant","Gene","Variant_type"]

    save_path = "/Users/malvikatejura/Downloads"
    
    file_name2 = "clinvar_wg_hg19_10152024" + ".csv"
    
    saved = os.path.join(save_path, file_name2)


    with open(saved,"w",encoding = "UTF8", newline = '') as new_file:
        writer = csv.writer(new_file)
        writer.writerow(header)
        for element in final_list:
            writer.writerow(element)
            
            
    df1 = pd.read_csv(saved, sep = ",", header = 0, dtype = {"Chromosome": str , "Genomic_coordinates" : int})
    
    df1.to_csv(path_or_buf= file_name2)
            
clinvar_38("/Users/malvikatejura/Downloads/clinvar_10152024_hg19.vcf.gz")

KeyboardInterrupt: 

In [67]:
fh = pd.read_csv("~/pillar_data_clinvar38_test_annotated.csv")

lh = pd.read_csv("~/clinvar_wg_hg19_10152024.csv")

/tmp/6075162.1.fowler-login.q/ipykernel_4187203/3241519560.py:1: DtypeWarning: Columns (3,9,11,12,13,16,17,20,21,32,35,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  fh = pd.read_csv("~/pillar_data_clinvar38_test_annotated.csv")
/tmp/6075162.1.fowler-login.q/ipykernel_4187203/3241519560.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  lh = pd.read_csv("~/clinvar_wg_hg19_10152024.csv")


In [68]:
merged_2 = pd.merge(fh, lh, left_on=['Gene', 'ref_allele','alt_allele','hg19_pos'], right_on = ['Gene',"Ref_allele",
                                   "Alt_allele",'Genomic_coordinates'], how='left')

columns_to_update_2 = ['clinvar_sig']
columns_with_fallback_2 = ['Significance']

for x_col, y_col in zip(columns_to_update_1, columns_with_fallback_1):
    merged_2[x_col] = merged_2[x_col].fillna(merged_2[y_col])

merged_2.drop(columns=['Chromosome', 'Genomic_coordinates', 'Ref_allele',
       'Alt_allele', 'Variant', 'Variant_type','Unnamed: 0',"Significance"], inplace=True)

merged_2.to_csv("~/pillar_data_clinvar38_19_annotated_final.csv", index = False)

In [75]:
merged_2

,Dataset,Gene,HGNC_id,Chrom,hg19_pos,hg38_start,hg38_end,ref_allele,alt_allele,auth_transcript_id,transcript_pos,transcript_ref,transcript_alt,aa_pos,aa_ref,aa_alt,hgvs_c,hgvs_p,consequence,auth_reported_score,auth_reported_rep_score,auth_reported_func_class,auth_reported_normal_min,auth_reported_normal_max,auth_reported_abnormal_min,auth_reported_abnormal_max,splice_measure,gnomad_MAF,clinvar_sig,clinvar_star,clinvar_date_last_reviewed,nucleotide_or_aa,MaveDB URN,Ensembl_transript_ID,Ref_seq_transcript_ID,Model_system,Assay_type,Phenotype_measured,Phenotype_detail,IGVF_produced
0,BRCA1_Findlay_2018,BRCA1,1100,17,41276135.0,NaN,NaN,T,G,NM_007294.3,-19-3,A,C,NaN,NaN,NaN,NaN,NaN,Splice region,-0.372611,-0.568675758504309;-0.176545248348852,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,Likely_benign,NaN,NaN,nucleotide,urn:mavedb:00000097-0-1,ENST00000357654.9,NM_007294.3,immortalized human cells,Cell Viability,Cell Survival,Overall function,No
1,BRCA1_Findlay_2018,BRCA1,1100,17,41276135.0,NaN,NaN,T,C,NM_007294.3,-19-3,A,G,NaN,NaN,NaN,NaN,NaN,Splice region,-0.045313,-0.332667114078832;0.242040175629815,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,Benign,NaN,NaN,nucleotide,urn:mavedb:00000097-0-1,ENST00000357654.9,NM_007294.3,immortalized human cells,Cell Viability,Cell Survival,Overall function,No
2,BRCA1_Findlay_2018,BRCA1,1100,17,41276135.0,NaN,NaN,T,A,NM_007294.3,-19-3,A,T,NaN,NaN,NaN,NaN,NaN,Splice region,-0.108254,-0.440230467981572;0.223722191334123,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,not_provided,NaN,NaN,nucleotide,urn:mavedb:00000097-0-1,ENST00000357654.9,NM_007294.3,immortalized human cells,Cell Viability,Cell Survival,Overall function,No
3,BRCA1_Findlay_2018,BRCA1,1100,17,41276134.0,NaN,NaN,T,G,NM_007294.3,-19-2,A,C,NaN,NaN,NaN,NaN,NaN,Canonical splice,-0.277963,-0.41057727764695;-0.145348986773218,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,not_provided,NaN,NaN,nucleotide,urn:mavedb:00000097-0-1,ENST00000357654.9,NM_007294.3,immortalized human cells,Cell Viability,Cell Survival,Overall function,No
4,BRCA1_Findlay_2018,BRCA1,1100,17,41276134.0,NaN,NaN,T,C,NM_007294.3,-19-2,A,G,NaN,NaN,NaN,NaN,NaN,Canonical splice,-0.388414,-0.6350191556350679;-0.141808664620659,FUNC,-0.748,1.307,-5.651,-1.328,Yes,NaN,Pathogenic,NaN,NaN,nucleotide,urn:mavedb:00000097-0-1,ENST00000357654.9,NM_007294.3,immortalized human cells,Cell Viability,Cell Survival,Overall function,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
650743,CTCF_unpublished,CTCF,13723,16,NaN,67626643.0,67626643.0,C,G,NaN,1446,NaN,NaN,482,L,L,c.1446C>G,NaN,synonymous_variant,0.959996,-0.6541320779744095;0.1299313120361334;0.23398...,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,nucleotide,NaN,ENST00000264010.10,NM_006565.4,immortalized human cells,Cell viability,Cell survival,overall function,Yes
650744,CTCF_unpublished,CTCF,13723,16,NaN,67626643.0,67626643.0,C,T,NaN,1446,NaN,NaN,482,L,L,c.1446C>T,NaN,synonymous_variant,0.901529,-0.157539930598156;-0.0646627700816877;0.28777...,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,nucleotide,NaN,ENST00000264010.10,NM_006565.4,immortalized human cells,Cell viability,Cell survival,overall function,Yes
650745,CTCF_unpublished,CTCF,13723,16,NaN,67626644.0,67626644.0,A,C,NaN,1447,NaN,NaN,483,I,L,c.1447A>C,NaN,missense_variant,1.040071,0.3758153232205121;0.6194697060103307;0.396447...,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,nucleotide,NaN,ENST00000264010.10,NM_006565.4,immortalized human cells,Cell viability,Cell survival,overall function,Yes
650746,CTCF_unpublished,CTCF,13723,16,NaN,67626644.0,67626644.0,A,G,NaN,1447,NaN,NaN,483,I,V,c.1447A>G,NaN,missense_variant,0.942589,-0.3943383820371419;0.0719964942613049;0.60524...,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,nucleotide,NaN,ENST00000264010.10,NM_006565.4,immortalized human cells,Cell viability,Cell survival,overall function,Yes


In [13]:
import pandas as pd

merged_2 = pd.read_csv("~/pillar_data_clinvar38_19_annotated_final.csv")

group_cols = ['Dataset', 'hgvs_p', 'auth_reported_score']

merged_2[group_cols] = merged_2[group_cols].fillna('MISSING')

condensed_df = (
    merged_2.groupby(group_cols)
    .agg(lambda x: ';'.join(map(str, x.dropna())))  # Skip NaNs when condensing
    .reset_index()
)

condensed_df[group_cols] = condensed_df[group_cols].replace('MISSING', pd.NA)



/tmp/6075162.1.fowler-login.q/ipykernel_26774/1015184770.py:3: DtypeWarning: Columns (3,9,11,12,13,16,17,20,21,32,35,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_2 = pd.read_csv("~/pillar_data_clinvar38_19_annotated_final.csv")


159916


In [16]:
condensed_df.to_csv("~/pillar_data_harmonized_v2.csv", index = False)